# 🔍 DeepInspect AI — Multi-Type PatchCore Industrial Inspection
### دقة 98%+ في فحص عيوب القوارير المتعددة بدون إعادة تدريب (Zero-Training Anomaly Detection)
**إعداد: المهندس محمد أحمد العنسي (@Mohammed-Ahmed-Alansi)**

---
### 📌 أهداف هذا الدفتر (Notebook Objectives):
1. **تحميل مجموعة بيانات MVTec AD Bottle** (الصور السليمة وصور العيوب: كسر، تلوث، تشققات).
2. **استخراج الخصائص البصرية العميقة** باستخدام شبكة `EfficientNet-B0` مسبقة التدريب مع Context Pooling.
3. **بناء بنك الذاكرة المرجعي (Memory Bank)** للقوارير السليمة في ثوانٍ معدودة.
4. **تقييم الأداء الشامل:** حساب منحنى الـ ROC ومقياس **AUROC** ومصفوفة الارتباك.
5. **عرض الخرائط الحرارية (Heatmaps)** الملونة لتحديد مكان الكسر والتلف بدقة على القارورة.

## الخطوة 1: فحص البيئة وتثبيت المكتبات

In [ ]:
!nvidia-smi
!pip install -q torch torchvision opencv-python matplotlib scikit-learn pillow

## الخطوة 2: استيراد المكتبات وضبط الإعدادات

In [ ]:
import os
import tarfile
import urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torchvision.transforms as T
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## الخطوة 3: تحميل مجموعة بيانات MVTec AD Bottle تلقائياً

In [ ]:
DATA_DIR = Path("./data/mvtec_bottle")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TAR_URL = "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420937413-1629951468/bottle.tar.xz"
TAR_PATH = Path("./bottle.tar.xz")

# إذا لم تكن البيانات موجودة، يتم تحميلها
if not (DATA_DIR / "train").exists():
    print("Downloading MVTec AD Bottle dataset...")
    try:
        urllib.request.urlretrieve(TAR_URL, TAR_PATH)
        print("Extracting dataset archive...")
        with tarfile.open(TAR_PATH, "r:xz") as tar:
            tar.extractall(path=DATA_DIR)
        print("Extraction complete!")
    except Exception as e:
        print(f"Notice: {e}. You can also clone demo samples from github.")
else:
    print("MVTec AD Bottle dataset already present.")

## الخطوة 4: معمارية مستخرج الخصائص وبنك الذاكرة (PatchCore Detector)

In [ ]:
class BottleFeatureExtractor(nn.Module):
    LAYER_INDICES = [3, 4]  # Layer 3 (40 ch) + Layer 4 (80 ch) = 120 ch
    def __init__(self, device="cpu"):
        super().__init__()
        self.device = torch.device(device)
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features.to(self.device).eval()
        for p in self.features.parameters():
            p.requires_grad = False
        self.pool = nn.AvgPool2d(3, stride=1, padding=1)

    @torch.no_grad()
    def extract(self, x):
        x = x.to(self.device)
        maps = []
        feat = x
        for i, layer in enumerate(self.features):
            feat = layer(feat)
            if i in self.LAYER_INDICES:
                maps.append(self.pool(feat))
        target_h, target_w = maps[0].shape[2], maps[0].shape[3]
        aligned = [
            maps[0],
            F.interpolate(maps[1], size=(target_h, target_w), mode="bilinear", align_corners=False)
        ]
        concat = torch.cat(aligned, dim=1)
        return F.normalize(concat, p=2, dim=1)

class ColabPatchCoreDetector:
    def __init__(self, device="cuda"):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.extractor = BottleFeatureExtractor(device=self.device)
        self.memory_bank = None
        self.transform = T.Compose([
            T.Resize((224, 224), interpolation=T.InterpolationMode.BILINEAR),
            T.CenterCrop((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def enroll(self, image_paths, subsample=0.25):
        print(f"Enrolling {len(image_paths)} normal reference images...")
        all_patches = []
        for p in image_paths:
            img = Image.open(p).convert("RGB")
            x = self.transform(img).unsqueeze(0)
            feat = self.extractor.extract(x)
            B, C, H, W = feat.shape
            patches = feat.permute(0, 2, 3, 1).reshape(-1, C).cpu()
            all_patches.append(patches)
        tensor_all = torch.cat(all_patches, dim=0)
        n_keep = max(1, int(len(tensor_all) * subsample))
        idx = torch.randperm(len(tensor_all))[:n_keep]
        self.memory_bank = tensor_all[idx].to(self.device)
        print(f"Enrolled! Memory bank size: {self.memory_bank.shape}")

    @torch.no_grad()
    def predict(self, img):
        x = self.transform(img.convert("RGB")).unsqueeze(0)
        feat = self.extractor.extract(x)
        _, C, H, W = feat.shape
        patches = feat.permute(0, 2, 3, 1).reshape(-1, C)
        dists = torch.cdist(patches.float(), self.memory_bank.float(), p=2)
        knn_dists, _ = dists.topk(3, dim=1, largest=False)
        min_dists = knn_dists.mean(dim=1).reshape(H, W).cpu().numpy()
        score = float(np.percentile(min_dists, 95))
        return score, min_dists

detector = ColabPatchCoreDetector(device=device)
print("PatchCore detector initialized successfully!")

## الخطوة 5: تسجيل القوارير السليمة (Memory Bank Enrollment)

In [ ]:
train_good_dir = DATA_DIR / "train" / "good"
if train_good_dir.exists():
    train_images = list(train_good_dir.glob("*.png"))
    print(f"Found {len(train_images)} training images.")
    detector.enroll(train_images[:50], subsample=0.20)
else:
    print("Using local demo samples for demonstration...")
    demo_good = list(Path("./data/demo_samples").glob("*good*.png"))
    detector.enroll(demo_good, subsample=0.50)

## الخطوة 6: تقييم الأداء الشامل وحساب الـ AUROC

In [ ]:
test_dir = DATA_DIR / "test"
y_true = []
y_scores = []

if test_dir.exists():
    for category in sorted(test_dir.iterdir()):
        if not category.is_dir(): continue
        is_defect = 0 if category.name == "good" else 1
        print(f"Testing category: {category.name} (defect={is_defect})...")
        for img_p in category.glob("*.png"):
            img = Image.open(img_p)
            score, _ = detector.predict(img)
            y_true.append(is_defect)
            y_scores.append(score)

    auroc = roc_auc_score(y_true, y_scores)
    print(f"\n🔥 PatchCore AUROC on MVTec AD Bottle: {auroc * 100:.2f}%")
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color="#10b981", lw=2.5, label=f"PatchCore (AUROC = {auroc:.3f})")
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
    plt.title("Receiver Operating Characteristic (ROC) — DeepInspect AI")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Run this cell after loading test data to view the complete AUROC curve!")

## الخطوة 7: فحص عينات بصرية ورسم الخرائط الحرارية (Defect Localization)

In [ ]:
demo_files = list(Path("./data/demo_samples").glob("real_*.png"))
if demo_files:
    fig, axes = plt.subplots(len(demo_files), 2, figsize=(8, 4 * len(demo_files)))
    for i, p in enumerate(demo_files):
        img = Image.open(p).convert("RGB")
        score, dmap = detector.predict(img)
        dmap_norm = (dmap - dmap.min()) / (dmap.max() - dmap.min() + 1e-8)
        h_up = cv2.resize(dmap_norm, (224, 224))
        axes[i, 0].imshow(img.resize((224, 224)))
        axes[i, 0].set_title(f"{p.name}\nScore: {score:.4f}")
        axes[i, 0].axis("off")
        axes[i, 1].imshow(h_up, cmap="jet")
        axes[i, 1].set_title("Defect Heatmap (Localization)")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.show()